# SmolVLM2 vision-transfer confirmatory sweep

This notebook runs the production-budget paired three-seed comparison only after the one-seed Smol pilot has sealed execution attestations, passed every required gate, and shown a positive heldout screening effect. It requires native CUDA BF16, such as L4, A10, A100, or newer. The final cells produce compact heldout-quality and multiplicity-controlled promotion evidence without printing per-sample outputs or long tables.

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

ROOT = next((p for p in (Path.cwd(), Path.cwd().parent) if (p / 'pyproject.toml').is_file()), None)
if ROOT is None:
    subprocess.run(['git', 'clone', 'https://github.com/SangbumChoi/OCR.git'], check=True)
    ROOT = Path('OCR').resolve()
subprocess.run(['git', '-C', str(ROOT), 'fetch', 'origin', 'claude/new-session-w79q0i'], check=True)
subprocess.run(['git', '-C', str(ROOT), 'checkout', 'claude/new-session-w79q0i'], check=True)
subprocess.run(['git', '-C', str(ROOT), 'merge', '--ff-only', 'origin/claude/new-session-w79q0i'], check=True)
os.chdir(ROOT)
if shutil.which('apt-get'):
    subprocess.run(['apt-get', 'update', '-qq'], check=True)
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'libpango-1.0-0', 'libpangoft2-1.0-0', 'libharfbuzz0b', 'libfontconfig1', 'fonts-liberation', 'fonts-noto-core', 'fonts-noto-cjk'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[student,student-gpu,newvlms,synth,finetune]'], check=True)
print('repo:', ROOT)


In [ ]:
import wandb

if not os.environ.get('WANDB_API_KEY'):
    try:
        from google.colab import userdata
        key = userdata.get('WANDB_API_KEY')
        if key:
            os.environ['WANDB_API_KEY'] = key
    except Exception:
        pass
wandb.login()
print('W&B target: https://wandb.ai/sbdc/docvlm-ablation')


The pilot output directory must be present in this runtime. Run the pilot notebook first or restore its complete `outputs/sweeps/docvlm-smol-vision-transfer-pilot` directory before continuing.

In [ ]:
import json

subprocess.run([sys.executable, 'scripts/audit_smol_vision_transfer_pilot_execution.py'], check=True)
subprocess.run([sys.executable, 'scripts/audit_smol_confirmatory_submission.py'], check=True)
gate = json.loads(Path('docs/results/smol_vision_confirmatory_submission.json').read_text())
print(json.dumps({
    'status': gate['overall_status'],
    'counts': gate['counts'],
    'authorized': gate['confirmatory_submission_authorized'],
    'fingerprint': gate['fingerprint'],
}, indent=2))
if not gate['confirmatory_submission_authorized']:
    raise RuntimeError('confirmatory submission is not authorized by sealed positive pilot evidence')


In [ ]:
# Compile all six runs and validate the environment without allocating model weights.
subprocess.run([sys.executable, 'scripts/run_smol_confirmatory_colab.py', '--dry-run', '--poll-seconds', '0.1'], check=True)


In [ ]:
# Run two arms across three paired seeds. Resume skips signature-valid completed stages.
subprocess.run([sys.executable, 'scripts/run_smol_confirmatory_colab.py'], check=True)


In [ ]:
subprocess.run([sys.executable, 'scripts/build_smol_confirmatory_evidence.py'], check=True)
quality = Path('docs/results/smol_vision_heldout_quality_evidence.json')
promotion = Path('docs/results/smol_vision_multi_seed_promotion_evidence.json')
subprocess.run([
    sys.executable,
    'scripts/audit_end_to_end_goal_readiness.py',
    '--quality-evidence', str(quality),
    '--promotion-evidence', str(promotion),
], check=True)
for path in (quality, promotion, Path('docs/results/end_to_end_goal_readiness.json')):
    payload = json.loads(path.read_text())
    print(json.dumps({
        'artifact': path.name,
        'claim_scope': payload.get('claim_scope'),
        'authorized': payload.get('quality_claim_authorized', payload.get('promotion_claim_authorized')),
        'status': payload.get('overall_status', payload.get('gate_status', payload.get('promotion_status'))),
        'fingerprint': payload.get('fingerprint'),
    }, sort_keys=True))
